# Crude MC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Parameters
s0 = 100
K = 110
H = 95
logH = np.log(H)
sigma = 0.25
sigmaJump = 0.1
m = 1.005
lam =.1
T = 1.0
r = .05
mu = r + lam*(1-m)
nu = mu - 0.5 * sigma**2
epsilon = 1e-8

# Returns number of jumps
def Poisson(lam):
  u = np.random.uniform(0,1)
  i = 0
  p = np.exp(-lam)
  while u > p:
    p = lam*p/(i+1)
    i += 1
  return i

def CallOptionBarrier(N):
    n_steps = 1000
    dt = T / n_steps
    call = np.zeros(N)
    call[0] = 0
    totalJumps = []

    for i in range(N):
        xt = np.log(s0)
        alive = True
        avgJumps = 0
        for _ in range(n_steps):
            njumps = np.random.poisson(lam * dt)
            jump_sum = 0
            if njumps > 0:
              avgJumps += avgJumps
              jump_sum = np.log(m) - 0.5 * sigmaJump**2 + sigmaJump * np.random.normal()

            xt += (nu)*dt + sigma*np.sqrt(dt)*np.random.normal() + jump_sum

            if xt <= logH:
                alive = False

        totalJumps.append(avgJumps)
        if alive:
            call[i] = max(np.exp(xt) - K, 0)
        else:
            call[i] = 0
    payoff = np.mean(call)
    std = np.std(call)
    avgJ = np.mean(avgJumps)
    return payoff, std, avgJ

avg_pay = []
avg_std = []
avJump = []
for i in range(30):
  print(f"Trial {i}")
  payoff, std, avgJ = CallOptionBarrier(1000)
  avg_pay.append(np.mean(payoff))
  avg_std.append(std)
  avJump.append(np.mean(avgJ))

print("Price: ", np.mean(avg_pay))
print("Std: ", np.mean(avg_std))
print("Avg Jumps per trial: ", np.mean(avJump))

Trial 0
Trial 1
Trial 2
Trial 3
Trial 4
Trial 5
Trial 6
Trial 7
Trial 8
Trial 9
Trial 10
Trial 11
Trial 12
Trial 13
Trial 14
Trial 15
Trial 16
Trial 17
Trial 18
Trial 19
Trial 20
Trial 21
Trial 22
Trial 23
Trial 24
Trial 25
Trial 26
Trial 27
Trial 28
Trial 29
Price:  5.1224376075946525
Std:  14.384940919378813
Avg Jumps per trial:  0.0


# Metwally and Atiya

In [ ]:
import numpy as np

s0 = 100
K = 110
H = 95
sigma = 0.25
sigmaJump = 0.1
lam = .1

T = 1.0
r = 0.05
m =  1.005
R =  0
# nu = r - lam*(m-1) - 0.5*sigma**2
# c = nu
logH = np.log(H)

def ITMexponential(lam):
    u = np.random.uniform(0,1)
    return -np.log(u)/lam

def GetJumps(lam):
    t = 0
    jumpTimes = []

    while True:
        tau = ITMexponential(lam)
        t += tau
        if t > T:
            break
        jumpTimes.append(t)
    return jumpTimes

def g_density(u, Ti_minus_one, Ti, Xt_minus, Xt_plus, logH, sigma, c):

    Tau = Ti - Ti_minus_one
    gamma = (1/(sigma*np.sqrt(2*np.pi*Tau))) * np.exp(
        -(Xt_plus - Xt_minus + c*Tau)**2/(2*sigma**2*Tau)) + 1e-8

    numerator = (Xt_plus - logH)
    denominator = 2*gamma*np.pi*sigma**2 * ((u-Ti_minus_one)**1.5) * ((Ti-u)**0.5)

    term1 = (Xt_minus - logH - c*(Ti-u))**2/(2*sigma**2*(Ti-u))
    term2 = (Xt_plus - logH + c*(u-Ti_minus_one))**2/(2*sigma**2*(u-Ti_minus_one))

    return (numerator/denominator) * np.exp(-(term1 + term2))


def MetwallyAtiya(lam, c):
    jumpTimes = GetJumps(lam)
    jumpTimes = [0] + jumpTimes + [T]
    logS = np.log(s0)


    for i in range(1, len(jumpTimes)):
        Ti_minus_one = jumpTimes[i-1]
        Ti = jumpTimes[i]
        Tau = Ti - Ti_minus_one

        St_prev = logS

        logS = St_prev + c*Tau + sigma*np.sqrt(Tau)*np.random.normal()
        prejumplogS=logS

        if i < len(jumpTimes)-1:
            logJ = np.log(m) - 0.5*sigmaJump**2 + sigmaJump*np.random.normal()
            logS += logJ
        if prejumplogS <= logH:
               payoff = R*np.exp(-r*Ti_minus_one)
               return payoff*0
        if logS <= logH:
               payoff = R*np.exp(-r*Ti_minus_one)
               return payoff*0
        if prejumplogS > logH and St_prev > logH:
            numerator = 2*(logH - St_prev)*(logH - prejumplogS)
            denominator = sigma**2 * Tau
            P =  1-np.exp(-numerator/denominator)
            P = min(P, .99999)
        else:
            P = 0

        b = Tau/(1-P)
        u = np.random.uniform(Ti_minus_one, Ti_minus_one + b)
        if u >= Ti_minus_one and u <= Ti:
            payoff = R*b*g_density(u, Ti_minus_one, Ti, St_prev, logS, logH, sigma, c)*np.exp(-r*u)
            return payoff*0


        if logS <= logH:
          payoff = R*np.exp(-r*Ti_minus_one)
          return payoff*0


    ST = np.exp(logS)
    return np.exp(-r*T) * max(ST - K, 0)


results = []
stds = []

nu = r + lam*(1-m) - .5*sigma**2
c = nu
for _ in range(30):
  trialPayoff = []
  for _ in range(10000):
    payoff = MetwallyAtiya(lam, c)
    trialPayoff.append(payoff)
  results.append(np.mean(trialPayoff))
  stds.append(np.std(trialPayoff, ddof=1))

print("Final price:", np.mean(results))
print("Std:", np.mean(stds))


# Joshi and Leung

In [ ]:
import numpy as np
import math
from scipy.stats import norm

s0 = 100
K = 110
H = 95
logH = np.log(H)
sigma = 0.25
sigmaJump = 0.1
m = 1.005
lam =8
T = 1.0
r = .05
PJump = 1 - np.exp(-lam*T)
PNoJump = np.exp(-lam*T)
mu = r + lam*(1-m)
nu = mu - 0.5 * sigma**2
q = math.ceil(lam*T+3*np.sqrt(lam*T))
print(f"q: {q}")

def Case1(lam):
  nu = r - lam * (m - 1) - 0.5 * sigma**2
  nu_tilde = nu +sigma**2
  d1 = (nu_tilde*T + np.log(s0/K))/(sigma*np.sqrt(T))
  d2 = (np.log(H**2/(K*s0))+nu_tilde*T)/(sigma*np.sqrt(T))
  d3 = (nu*T + np.log(s0/K))/(sigma*np.sqrt(T))
  d4 = (np.log(H**2/(K*s0))+nu*T)/(sigma*np.sqrt(T))
  term1 = s0*np.exp(lam*(1-m)*T)*(norm.cdf(d1) - (H/s0)**(2*nu_tilde*sigma**(-2))*norm.cdf(d2))
  term2 = -np.exp(-r*T)*K*(norm.cdf(d3)- (H/s0)**(2*nu*sigma**(-2))*norm.cdf(d4))
  return term1 + term2

def Xigen(i, means, variances, observedvals, w, n):
    conditionalmean = means[i] + (
        variances[i] * (w - np.sum(observedvals[0:i]) - np.sum(means[i:n])) /
        np.sum(variances[i:n])
    )
    conditionalvariance = variances[i] * (
        1 - variances[i] / np.sum(variances[i:n])
    )
    Z = np.random.normal(conditionalmean, np.sqrt(conditionalvariance))
    return Z


# def Xigen(i,means,variances,observedvals,w,n):
#   conditionalmean=means[i]+(variances[i]*(w-np.sum(observedvals[0:i])-np.sum(means[i:n]))/(np.sum(variances[i:n])))
#   conditionalvariance=variances[i]*(1-variances[i]/(np.sum(variances[i:n])))
#   Z=np.random.normal(conditionalmean,np.sqrt(conditionalvariance))
#   return Z

def Case2(Nt):
    J = []
    mesh = [] # jump times
    for i in range(Nt):
      J.append(m * np.exp(-.5 * sigmaJump**2 + sigmaJump * np.random.normal()))
      mesh.append(np.random.uniform(0,1))

    mesh = np.sort(mesh)
    mesh = [0] + [T * t for t in mesh] + [T]
    Jpr = np.prod(J)

    W = np.random.normal(nu * T, sigma * np.sqrt(T))
    while(W<=np.log(K/(s0*Jpr))):
       W = np.random.normal(nu * T, sigma * np.sqrt(T))


    ST = s0*Jpr*np.exp(W)
    stocks = []
    stocks.append(s0)
    Splus = s0
    observedvals = np.zeros(Nt)

    means = []
    var = []
    for i in range(Nt):
      means.append(nu*(mesh[i+1]-mesh[i]))
      var.append(sigma**2*(mesh[i+1]-mesh[i]))

    alphaPr = 1
    for i in range(Nt):
      xi = Xigen(i, means, var, observedvals, W,Nt)
      observedvals[i]=xi
      Sminus = Splus*np.exp(xi)
      stocks.append(Sminus)
      Splusold=Splus
      Splus = J[i]*Sminus
      stocks.append(Splus)
      if Sminus < H or Splus < H:
        return 0

      alpha =  np.exp(-2 * (logH- np.log(Splusold)) * (logH - np.log(Sminus))  / (sigma**2*(mesh[i+1] - mesh[i])))
      alphaPr *= 1-alpha
      #print(alpha)
    alpha= np.exp(-2 * (logH- np.log(Splus)) * (logH - np.log(ST))  / (sigma**2*(T- mesh[-2])))
    #print(alpha)
    alphaPr*=1-alpha
    Cjump = np.exp(-r*T)*max(ST-K,0)*(1-norm.cdf((np.log(K/(s0*Jpr))-nu*T)/(sigma*np.sqrt(T))))*alphaPr#*(1-norm.cdf((np.log(H/(s0*Jpr))-nu*T)/(sigma*np.sqrt(T))))

    return Cjump

def Case3():
    x = np.random.poisson(lam*T)
    while x <= q:
        x = np.random.poisson(lam*T)
    lastJump = Case2(x)
    return lastJump


def RossandGhamami():
  expected = 0
  sum_p=0
  for i in range(q):
    P = np.exp(-lam*T)*(lam*T)**i/math.factorial(i)
    sum_p+=P
    if i == 0:
      expected += Case1(i)*P

    if i > 0 and i < q:
      expected += Case2(i)*P

  prob=1-sum_p
  expected += Case3()*prob
  return expected


payoffs = [RossandGhamami() for i in range(500)]
print(np.mean(payoffs))
print(np.mean(np.std(payoffs)))
payoffs=[]
#for i in range(10000):
#  payoffs.append(Case2(8))
#print(np.mean(payoffs))

# Ross and Ghamami

In [ ]:
from re import L
import numpy as np
import math
from scipy.stats import norm

s0 = 100
K = 110
H = 95
logH = np.log(H)
sigma = 0.25
sigmaJump = 0.1
m = 1.005
lam =.1
T = 1.0
r = .05
PJump = 1 - np.exp(-lam*T)
PNoJump = np.exp(-lam*T)
mu = r + lam*(1-m)
nu = mu - 0.5 * sigma**2
q = math.ceil(lam*T+3*np.sqrt(lam*T))
print(f"q: {q}")

def ITMTruncatedNormal(threshold):
  theta = 1 - norm.cdf(threshold)
  u = np.random.uniform(0,1)
  z = norm.ppf(1-theta*u)
  return z, theta

def Case1(lam):
  nu = r - lam * (m - 1) - 0.5 * sigma**2
  nu_tilde = nu +sigma**2
  d1 = (nu_tilde*T + np.log(s0/K))/(sigma*np.sqrt(T))
  d2 = (np.log(H**2/(K*s0))+nu_tilde*T)/(sigma*np.sqrt(T))
  d3 = (nu*T + np.log(s0/K))/(sigma*np.sqrt(T))
  d4 = (np.log(H**2/(K*s0))+nu*T)/(sigma*np.sqrt(T))
  term1 = s0*np.exp(lam*(1-m)*T)*(norm.cdf(d1) - (H/s0)**(2*nu_tilde*sigma**(-2))*norm.cdf(d2))
  term2 = -np.exp(-r*T)*K*(norm.cdf(d3)- (H/s0)**(2*nu*sigma**(-2))*norm.cdf(d4))
  return term1 + term2

def Xigen(i, means, variances, observedvals, w, n):
    conditionalmean = means[i] + (
        variances[i] * (w - np.sum(observedvals[0:i]) - np.sum(means[i:n])) /
        np.sum(variances[i:n])
    )
    conditionalvariance = variances[i] * (
        1 - variances[i] / np.sum(variances[i:n])
    )
    Z = np.random.normal(conditionalmean, np.sqrt(conditionalvariance))
    return Z



def Case2(Nt):
    J = []
    mesh = [] # jump times
    for i in range(Nt):
      J.append(m * np.exp(-.5 * sigmaJump**2 + sigmaJump * np.random.normal()))
      mesh.append(np.random.uniform(0,1))

    mesh = np.sort(mesh)
    mesh = [0] + [T * t for t in mesh] + [T]
    Jpr = np.prod(J)

    threshold = (np.log(K/(s0*Jpr)) - nu*T)/(sigma*np.sqrt(T))
    z, theta = ITMTruncatedNormal(threshold)
    W = nu*T + sigma*np.sqrt(T)*z




    ST = s0*Jpr*np.exp(W)
    stocks = []
    stocks.append(s0)
    Splus = s0
    observedvals = np.zeros(Nt)

    means = []
    var = []
    for i in range(Nt):
      means.append(nu*(mesh[i+1]-mesh[i]))
      var.append(sigma**2*(mesh[i+1]-mesh[i]))

    alphaPr = 1
    for i in range(Nt):
      xi = Xigen(i, means, var, observedvals, W,Nt)
      observedvals[i]=xi
      Sminus = Splus*np.exp(xi)
      stocks.append(Sminus)
      Splusold=Splus
      Splus = J[i]*Sminus
      stocks.append(Splus)
      if Sminus < H or Splus < H:
        return 0

      alpha =  np.exp(-2 * (logH- np.log(Splusold)) * (logH - np.log(Sminus))  / (sigma**2*(mesh[i+1] - mesh[i])))
      alphaPr *= 1-alpha
      #print(alpha)
    alpha= np.exp(-2 * (logH- np.log(Splus)) * (logH - np.log(ST))  / (sigma**2*(T- mesh[-2])))
    #print(alpha)
    alphaPr*=1-alpha
    Cjump = np.exp(-r*T)*max(ST-K,0)*theta*(alphaPr)

    return Cjump

def Case3():
    x = np.random.poisson(lam*T)
    while x <= q:
        x = np.random.poisson(lam*T)
    lastJump = Case2(x)
    return lastJump


def stratified_RossandGhamami(n):
    stratified_payoffs = []
    probs = [np.exp(-lam*T)*(lam*T)**i / math.factorial(i) for i in range(q)]
    P_tail = 1 - sum(probs)

    total_probs = probs + [P_tail]
    total_sum = sum(total_probs)
    normalized_probs = [p / total_sum for p in total_probs]


    runs_per_stratum = [int(n * p) for p in normalized_probs]

    while sum(runs_per_stratum) < n:
        runs_per_stratum[-1] += 1  #

    for _ in range(runs_per_stratum[0]):
        stratified_payoffs.append(Case1(0))

    for m in range(1, q):
        for _ in range(runs_per_stratum[m]):
            stratified_payoffs.append(Case2(m))

    for _ in range(runs_per_stratum[-1]):
        stratified_payoffs.append(Case3())

    # Average result
    return stratified_payoffs




totalMeans = []
totalStdds= []
n = 1000
for i in range(30):
  sol = stratified_RossandGhamami(n)
  totalMeans.append(np.mean(sol))
  totalStdds.append(np.std(sol))


print(np.mean(totalMeans))
print(np.mean(totalStdds))

